In [0]:
%sql
SELECT *
FROM healthcare_dev.bronze.diagnosis_raw;

In [0]:
# ============================================================
# SILVER LAYER — DIM_DIAGNOSIS
# Source: healthcare_dev.bronze.diagnosis_raw
# Target: healthcare_dev.silver.dim_diagnosis
# ============================================================

from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1. Configure Bronze source, Silver target, and checkpoint
# ------------------------------------------------------------

bronze_table = "healthcare_dev.bronze.diagnosis_raw"
silver_table = "healthcare_dev.silver.dim_diagnosis"

checkpoint_path = (
    "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/"
    "silver/dim_diagnosis/checkpoint/"
)

# ------------------------------------------------------------
# 2. Read Bronze using Structured Streaming
# ------------------------------------------------------------

df_diagnosis_bronze = spark.readStream.table(bronze_table)

# ------------------------------------------------------------
# 3. Apply Silver transformations
# ------------------------------------------------------------

df_diagnosis_clean = (
    df_diagnosis_bronze
    .dropDuplicates(["diagnosis_code"])
    .withColumn("load_timestamp", current_timestamp())
)

# ------------------------------------------------------------
# 4. Define Delta MERGE operation
# ------------------------------------------------------------

def merge_dim_diagnosis(batch_df, batch_id):

    if not spark.catalog.tableExists(silver_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        return

    dim_diagnosis = DeltaTable.forName(spark, silver_table)

    (
        dim_diagnosis.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.diagnosis_code = s.diagnosis_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# ------------------------------------------------------------
# 5. Execute incremental streaming ingestion
# ------------------------------------------------------------

query = (
    df_diagnosis_clean.writeStream
    .foreachBatch(merge_dim_diagnosis)
    .outputMode("update")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

query.awaitTermination()

print("Silver dim_diagnosis ingestion completed.")

# ------------------------------------------------------------
# 6. Verify Silver table
# ------------------------------------------------------------

display(
    spark.table("healthcare_dev.silver.dim_diagnosis")
)